# einops-rearrange-flatten — worked example 2: Unflatten a linear projection back into a 2D patch grid

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-rearrange-flatten`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The inverse of flattening — axis decomposition — uses a parenthesized group on the *input* side of the pattern. `'b (h w) d -> b h w d'` splits the flat sequence dimension into spatial height and width, restoring the 2D grid structure. You must supply the size of at least one decomposed axis as a keyword argument, since einops cannot infer the split from shape alone.

## Worked solution

After a Vision Transformer patch embedding, we have tokens of shape `(B=2, N=16, D=64)` where `N = H*W = 4*4`. We want to restore the 2D grid: `(B, H=4, W=4, D=64)`.

**Pattern:** `'b (h w) d -> b h w d'` with `h=4`.

**Step 1.** Einops sees `(h w)` on the left with `h=4` supplied. It infers `w = N / h = 16 / 4 = 4`.
**Step 2.** The `n` axis of size 16 is split into `h=4, w=4`.
**Step 3.** Output shape: `(2, 4, 4, 64)`.

Note: you must supply at least one of `h` or `w`; supplying both is fine and makes intent explicit.

In [ ]:
import torch as t
from einops import rearrange

t.manual_seed(42)
B, H, W, D = 2, 4, 4, 32
# Simulate flattened patch tokens from a ViT encoder
tokens = t.randn(B, H * W, D)

def restore_grid(tokens, H, W):
    return rearrange(tokens, 'b (h w) d -> b h w d', h=H, w=W)

grid = restore_grid(tokens, H, W)
print('Tokens shape:', tokens.shape)  # (2, 16, 32)
print('Grid shape:', grid.shape)      # (2, 4, 4, 32)
assert grid.shape == (B, H, W, D)

# Verify round-trip: flatten then unflatten is identity
reflattened = rearrange(grid, 'b h w d -> b (h w) d')
assert t.allclose(reflattened, tokens)
print('Round-trip identity:', True)